# SC4001 Phase 3, DANN against baseline with early stopping disabled

Accelerator must be **GPU (T4 x2 or P100)**. `src/train_dann.py` refuses CPU by design.
Code and data come from the attached dataset `isaacwjk21/sc4001-project`; no internet or installs.

## Why this kernel exists

The lambda ablation (kernel `sc4001-phase3-davis-lambda`, 9 runs) returned the same answer at
every lambda: test MSE 0.4949 at lambda 1.0, 0.4946 at 0.5 and 0.4930 at 0.1, against the
baseline's 0.383. A 10x range in lambda changing nothing is the tell that lambda was not what
produced the difference.

What produced it was early stopping. Every DANN run peaked at epoch 3 to 6 and stopped at 18 to
21; every baseline run peaked at 51 to 93 and stopped at 66 to 100. Validation MSE on this split
sits on a noisy plateau near 0.64 to 0.69 for roughly twenty epochs before descending: the
baseline is at 0.675 by epoch 5 and only reaches 0.486 by epoch 52. With `patience` 15 and
`min_delta` 1e-4, whether a run escapes that plateau depends on whether it happens to post a
better-than-min_delta epoch inside the window. All three baseline runs did. None of the nine
DANN runs did. The reported 0.11 MSE deficit is therefore epochs of training, not adaptation.

So the comparison could not distinguish "DANN hurts" from "DANN lost the early-stopping
lottery", and this kernel removes the lottery.

## Design

Both arms run the full 100 epochs with `--patience 100`, which disables early stopping, so each
arm receives identical compute and the best epoch is selected from `history.csv` afterwards.
DAVIS `split_cold_target`, seeds 4001/4002/4003, six runs:

- baseline, `src/train.py`
- DANN at `--lambda-max 1.0`, `--discriminator-input protein`

Lambda is not re-swept here. It demonstrably made no difference while training was truncated, so
re-testing it is only informative once this confound is gone; that belongs in the next quota
window.

`--patience` is new on both entry points and is saved into `config.json`, so these runs are
distinguishable from the patience-15 ones. Output goes under `outputs/nostop/<arm>/` because the
per-run path carries neither the patience nor the arm, and the patience-15 results at the
standard path must survive for comparison.

## What the output answers

The last two cells print, per arm and seed, the best validation MSE over all 100 epochs with the
epoch it occurred at, and the test metrics at each run's own best checkpoint. If DANN's best
validation MSE over 100 epochs is close to the baseline's, the earlier deficit was the stopping
rule and DANN is simply not helping; if DANN's best is still clearly worse at equal compute,
then the adversarial term genuinely damages cold-target regression, which is a real negative
result about the method on this problem.

Outputs survive only if this version is saved.


In [ ]:
import json, os, shutil, subprocess, sys, time
from pathlib import Path
import torch

if not torch.cuda.is_available():
    raise SystemExit("No CUDA device. Set the accelerator to GPU T4 x2 or P100, not TPU or None.")
props = torch.cuda.get_device_properties(0)
print("torch", torch.__version__, "| device", props.name,
      "|", round(props.total_memory / 1e9, 1), "GB | count", torch.cuda.device_count())


In [ ]:
# Show what is actually mounted, so a layout surprise is visible rather than guessed at.
inputs = sorted(Path("/kaggle/input").rglob("*"))
print(len(inputs), "mounted paths; first 25:")
for path in inputs[:25]:
    print("   ", path)

# Locate the project root: the directory holding both src/train.py and data/processed/vocab.json.
# Kaggle strips the archive's top-level folder, so do not assume a "Project" path segment.
candidates = [p.parents[1] for p in Path("/kaggle/input").rglob("src/train.py")]
roots = [p for p in candidates if (p / "data" / "processed" / "vocab.json").exists()]
if not roots:
    raise SystemExit(
        "No project root found. Looked for src/train.py beside data/processed/vocab.json. "
        "Candidates with src/train.py but no vocab.json: " + str(candidates)
    )
source = sorted(roots)[0]

project = Path("/kaggle/working/sc4001/Project")
shutil.copytree(source, project, dirs_exist_ok=True)
env = dict(os.environ, SC4001_OUTPUT_ROOT="/kaggle/working/outputs", PYTHONUNBUFFERED="1")
print("source :", source)
print("project:", project)
print("tables :", sorted(p.name for p in (project / "data" / "processed").iterdir()))

# Phase 3 needs the external kinase pool and the extra split columns, not just the Phase 2 tables.
required = ["external_kinases.csv", "external_tokens_davis.npz", "external_tokens_kiba.npz",
            "davis_extra_splits.csv", "kiba_extra_splits.csv"]
present = {p.name for p in (project / "data" / "processed").iterdir()}
missing = [name for name in required if name not in present and name + ".gz" not in present]
if missing:
    raise SystemExit("Dataset is missing Phase 3 inputs: " + str(missing))
print("phase 3 inputs OK")


In [ ]:
# Smoke the DANN entry point before spending quota: it has never run on a GPU.
smoke = subprocess.run(
    [sys.executable, "src/train_dann.py", "--dataset", "davis",
     "--split", "split_cold_target", "--smoke"],
    cwd=project, env=env, check=False,
)
if smoke.returncode != 0:
    raise SystemExit("DANN smoke run failed with code " + str(smoke.returncode)
                     + "; not starting the grid.")
for stale in Path("/kaggle/working/outputs").rglob("*-smoke"):
    shutil.rmtree(stale, ignore_errors=True)
print("smoke OK")


In [ ]:
# Both arms run the full 100 epochs with early stopping disabled (--patience 100), so the
# stopping rule cannot decide the comparison. The best epoch is selected from history.csv
# afterwards. Output goes under outputs/nostop/ because the per-run path carries neither the
# patience nor the arm, and the patience-15 results at the standard path must survive.
ROOT = "/kaggle/working/outputs/nostop"
GRID = ([("src/train.py", "baseline", seed, []) for seed in (4001, 4002, 4003)]
        + [("src/train_dann.py", "dann-lambda1.0", seed, ["--lambda-max", "1.0",
                                                          "--discriminator-input", "protein"])
           for seed in (4001, 4002, 4003)])

started = time.time()
status = []
for script, arm, seed, extra in GRID:
    t0 = time.time()
    label = arm + " davis split_cold_target seed" + str(seed)
    print()
    print("===== " + label + " (patience 100, no early stop) =====", flush=True)
    run = subprocess.run(
        [sys.executable, script, "--dataset", "davis", "--split", "split_cold_target",
         "--seed", str(seed), "--patience", "100", "--resume"] + extra,
        cwd=project, env=dict(env, SC4001_OUTPUT_ROOT=ROOT + "/" + arm), check=False,
    )
    elapsed = time.time() - t0
    status.append((label, run.returncode, elapsed))
    print("----- rc=" + str(run.returncode) + " wall=" + format(elapsed / 60, ".1f")
          + " min -----", flush=True)

print()
print("Grid wall time", format((time.time() - started) / 60, ".1f"), "min")
for label, rc, elapsed in status:
    print("%-52s rc=%d %6.1f min" % (label, rc, elapsed / 60))
failed = [row for row in status if row[1] != 0]
print("FAILED RUNS:", failed if failed else "none")

# The comparison this kernel exists to make: best validation MSE over the full 100 epochs,
# per arm and seed, with the epoch it occurred at. Equal compute, no stopping lottery.
print()
print("Best validation MSE over 100 epochs, by arm:")
for path in sorted(Path(ROOT).glob("**/history.csv")):
    rows = [line.split(",") for line in path.read_text().strip().splitlines()]
    header, body = rows[0], rows[1:]
    if not body:
        continue
    index = {name: position for position, name in enumerate(header)}
    best = min(body, key=lambda row: float(row[index["val_mse"]]))
    arm = str(path).replace(ROOT + "/", "").split("/")[0]
    print("  %-16s %-34s best val_mse %.6f at epoch %-4s (final epoch %s)"
          % (arm, str(path.parent).split("/")[-2] + "/" + str(path.parent).split("/")[-1],
             float(best[index["val_mse"]]), best[index["epoch"]], body[-1][index["epoch"]]))

print()
print("Test metrics at each run's own best checkpoint:")
for path in sorted(Path(ROOT).glob("**/metrics.json")):
    payload = json.loads(path.read_text())
    arm = str(path).replace(ROOT + "/", "").split("/")[0]
    print("  %-16s seed%d  test MSE %.4f  CI %.4f  best_ep %d  stop_ep %d  lambda %s"
          % (arm, payload["seed"], payload["test"]["all"]["mse"], payload["test"]["all"]["ci"],
             payload["best_epoch"], payload["stopped_epoch"], payload.get("lambda_max", "n/a")))


In [ ]:
# Print every metrics.json so the numbers live in the saved notebook even if files are lost.
found = sorted(Path("/kaggle/working/outputs").glob("**/metrics.json"))
print(len(found), "metrics files")
rows = []
for path in found:
    payload = json.loads(path.read_text())
    print("=" * 70)
    print(path)
    print(json.dumps(payload, indent=2))
    model = payload["test"]["all"]
    phase = "phase3" if "phase3" in str(path) else "phase2"
    rows.append((phase, payload["split"], payload["seed"], payload.get("lambda_max"),
                 model["mse"], model["ci"], payload["best_epoch"], payload["total_seconds"]))

print("=" * 100)
print("%-7s %-26s %5s %7s %9s %8s %8s %9s"
      % ("phase", "split", "seed", "lambda", "mse", "ci", "best_ep", "minutes"))
for phase, split, seed, lam, mse, ci, best_ep, secs in sorted(rows):
    lam_text = "    n/a" if lam is None else "%7.2f" % lam
    print("%-7s %-26s %5d %s %9.4f %8.4f %8d %9.1f"
          % (phase, split, seed, lam_text, mse, ci, best_ep, secs / 60))
